In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.12 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Y₂O₃ on SPODI

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit cubic Y₂O₃ to constant-wavelength neutron powder data from SPODI at FRM II, with anisotropic
displacement tensors whose components the site symmetry ties together.

The project is the catalogue case `pd-neut-cwl_y2o3-spodi`, the one the command line and the
app's Examples open.

In [2]:
import easydiffraction as edi

## Load the project

In [3]:
project_path = edi.download_data('pd-neut-cwl_y2o3-spodi', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['y2o3']
experiment = project.experiments['spodi']

Data 'pd-neut-cwl_y2o3-spodi' is at data/pd-neut-cwl_y2o3-spodi-6856c2525734/pd-neut-cwl_y2o3-spodi


The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [4]:
for parameter in project.free_parameters:
    parameter.free = False

## Estimate the background

Start the background from an automatic estimate. The default method, arPLS, fits a smooth curve
under the pattern that ignores the peaks, and places 21 points along it.

In [5]:
experiment.background.auto_estimate(n_points=21)

In [6]:
project.display.pattern('spodi')

View(html=<397,669 characters>, height=480)

## Perform the analysis

The fit goes in stages, freeing one group of parameters at a time.

### Scale, 2θ zero and cell

The scale, the zero shift and the cell length put the peaks in place and at the right height.

In [7]:
experiment.linked_structures['y2o3'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True

In [8]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,y2o3,cell,,length_a,10.61,,0,30,Å
2,spodi,instrument,,calib_twotheta_offset,0,,-inf,inf,deg
3,spodi,linked_structure,y2o3,scale,0.1,,-inf,inf,


In [9]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_y2o3-spodi, 2509 pts fitted / 3060 loaded, 3 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        3.47         7.84
      1      0.042      2.53         4.17   46.8% ↓
      2      0.050      2.53         4.17   
  done · converged · 2 iters · reduced χ² 4.17 · Rwp 2.53 % · 0.06 s


View(html=<84,278 characters>, height=376)

In [10]:
project.display.pattern('spodi')

View(html=<398,225 characters>, height=480)

### Background

The estimated points are a starting point; fit them now that the peaks are in place.

In [11]:
for point in experiment.background:
    point.intensity.free = True

In [12]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_y2o3-spodi, 2509 pts fitted / 3060 loaded, 24 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.53         4.20
      1      0.039      2.37         3.68   12.4% ↓
  done · converged · 1 iters · reduced χ² 3.68 · Rwp 2.37 % · 0.05 s


View(html=<112,625 characters>, height=376)

In [13]:
project.display.pattern('spodi')

View(html=<399,325 characters>, height=480)

### Peak widths

In [14]:
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True

In [15]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_y2o3-spodi, 2509 pts fitted / 3060 loaded, 27 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.37         3.69
      1      0.054      1.78         2.08   43.5% ↓
  done · converged · 1 iters · reduced χ² 2.08 · Rwp 1.78 % · 0.06 s


View(html=<117,417 characters>, height=376)

In [16]:
project.display.pattern('spodi')

View(html=<399,351 characters>, height=480)

### Atom positions

Y1's x and O1's three coordinates are the ones the symmetry leaves free.

In [17]:
for site in structure.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [18]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_y2o3-spodi, 2509 pts fitted / 3060 loaded, 31 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        1.78         2.09
      1      0.057      1.42         1.33   36.5% ↓
  done · converged · 1 iters · reduced χ² 1.33 · Rwp 1.42 % · 0.06 s


View(html=<121,621 characters>, height=376)

In [19]:
project.display.pattern('spodi')

View(html=<399,337 characters>, height=480)

### Anisotropic displacements

Last, the displacement tensors, each with only the components its site symmetry leaves free.

In [20]:
for tensor in structure.atom_site_aniso:
    for component in (
        tensor.adp_11,
        tensor.adp_22,
        tensor.adp_33,
        tensor.adp_12,
        tensor.adp_13,
        tensor.adp_23,
    ):
        component.free = not component.symmetry_constrained

In [21]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_y2o3-spodi, 2509 pts fitted / 3060 loaded, 43 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        1.42         1.33
      1      0.069      1.33         1.17   11.9% ↓
  done · converged · 1 iters · reduced χ² 1.17 · Rwp 1.33 % · 0.08 s


View(html=<136,652 characters>, height=376)

In [22]:
project.display.pattern('spodi')

View(html=<399,414 characters>, height=480)

In [23]:
project.display.fit.correlations()

,1,2,3,4,5
1 y2o3.cell.length_a,,,,,
2 spodi.instrument.calib_twotheta_offset,0.85,,,,
3 spodi.peak.broad_gauss_u,-0.00,-0.00,,,
4 spodi.peak.broad_gauss_v,0.00,0.01,-0.96,,
5 spodi.peak.broad_gauss_w,-0.00,-0.01,0.84,-0.94,


In [24]:
project.display.structure('y2o3')

View(html=<721,569 characters>, height=660)

## Save the project

In [25]:
project.save_as('pd-neut-cwl_y2o3-spodi')
project.report.save_html('pd-neut-cwl_y2o3-spodi.html')

Saving project to 'pd-neut-cwl_y2o3-spodi'
Saving report to 'pd-neut-cwl_y2o3-spodi.html'
